# Transformer estándar para el TSP (Glimpse)

## Generación del conjunto de datos

Como revisamos en el primer tutorial, el paso inicial para aplicar aprendizaje supervisado es construir nuestro conjunto de datos. Generaremos un dataset de 5000 ejemplos basados en instancias de 50 ciudades, utilizando los adaptadores por defecto para formatear las coordenadas de entrada y utilizar la siguiente ciudad del tour (según el algoritmo de resolución OR-Tools) como salida.

In [5]:
from instances.instances import generate_instances

instances = generate_instances(filename="TSP50.pkl", instance_count=1000, cities=50, seed=42)

from data.generation import generate_train_data
from data.adapters.input.default import DefaultInputAdapter
from data.adapters.output.default import DefaultOutputAdapter

input_config = (DefaultInputAdapter, 50)
output_config = (DefaultOutputAdapter, 50)

generate_train_data(
    instance_file="TSP50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    size=5000
)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 5000)


## Arquitectura Glimpse

Para resolver el TSP, necesitamos una arquitectura especializada que se adapte a la estructura del problema. Por una parte, necesitamos de un ***encoder*** capaz de capturar la estructura del grafo subyacente de ciudades. Por otra, necesitamos de un ***decoder*** que, dado este grafo codificado y el estado actual del tour, pueda tomar decisiones acertadas de la siguiente ciudad a visitar.

La arquitectura típica para resolver este problema se basa en mecanismos de tipo ***Glimpse***. Un diagrama de la arquitectura se muestra en la siguiente imagen:

<img src="../resources/glimpse.png" width="1000" alt="Arquitectura Glimpse para el TSP">

A continuación se explica en detalle cada componente:

### **Encoder (Extracción de características)**

Toma las coordenadas 2D originales y las expande a un espacio latente (`embed_dim`) mediante una proyección lineal. Luego, aplica múltiples capas de *Self-Attention* estándar. El resultado son los *City embeddings* (la "memoria" del modelo): una representación profunda donde cada ciudad ya no es solo una coordenada $(x, y)$, sino un vector rico en contexto que comprende su relación espacial con el resto de las ciudades del grafo.

### **Decoder (Construcción de la solución)**

El *decoder* se encuentra dividido en tres etapas principales:

1. **Representación del Tour**: El decodificador debe decidir qué ciudad visitar a continuación, pero para ello necesita entender el estado actual de la ruta. Lo hace construyendo un vector de contexto (*Query*) a partir de los *embeddings* del encoder basándose en tres elementos de las ciudades ya visitadas:

    - La **media** de todas las ciudades en la ruta actual (`Mean`).

    - La **primera ciudad** del recorrido, crucial para saber a dónde debe regresar al final (`First city`).

    - La **última ciudad** visitada, que representa la posición actual del vendedor (`Last city`).

2. **Mecanismo Glimpse (Cross-Attention)**:
Estos tres vectores se concatenan y se reducen para formar la *Query* inicial (Q). En lugar de calcular las probabilidades directamente, el modelo realiza uno o más "vistazos" (*Glimpses*). Utiliza la Query para consultar iterativamente los embeddings de las ciudades no visitadas (*Keys*), previamente obtenidas aplicando una máscara (`Mask`) a las ciudades visitadas. Esto permite a la red asimilar la distribución del resto del mapa y refinar su estado interno antes de tomar la decisión final.

3. **Pointer Scoring y Enmascaramiento**: El contexto refinado pasa por una última proyección para calcular el producto punto contra la memoria del encoder. Esto genera los *Logits* finales. Simultáneamente, el bloque `Mask` final inyecta un valor de infinito negativo ($-1\times 10^9$) a las posiciones de *padding* y a las ciudades ya visitadas, garantizando estructuralmente que el modelo nunca genere rutas inválidas.

A continuación, la implementación completa de esta arquitectura en PyTorch:

In [6]:
import torch
import torch.nn as nn
import math
from models.base.transformer import Transformer

class TSPTransformer(Transformer):
    def __init__(self, input_dim=2, embed_dim=128, num_heads=8, num_encoder_layers=2, num_glimpses=2, dropout_rate=0.1):
        super().__init__(
            input_dim=input_dim,
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_encoder_layers=num_encoder_layers,
            num_glimpses=num_glimpses,
            dropout_rate=dropout_rate,
        )
        self.embed_dim = embed_dim
        
        # --- ENCODER ---
        self.encoder_input_layer = nn.Linear(input_dim, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dropout=dropout_rate,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers, enable_nested_tensor=False)
        
        # --- DECODER ---
        self.ctx_fusion = nn.Linear(3 * embed_dim, embed_dim)

        self.num_glimpses = num_glimpses
        self.glimpse_proj = nn.Linear(embed_dim, embed_dim)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout_rate,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

        self.pointer_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def _get_pad_mask(self, max_len, num_cities, device):
        """
        Crea una máscara booleana de tamaño (Batch, max_len) donde True indica 
        que esa posición es padding y debe ser ignorada.
        """
        # Crea un tensor [0, 1, ..., max_len - 1] y lo expande
        idx = torch.arange(max_len, device=device).unsqueeze(0)
        # Compara con la cantidad real de ciudades para cada elemento del batch
        return idx >= num_cities.unsqueeze(1)

    # =====================================================
    # 1. --- ENCODER ---
    # =====================================================
    def encode(self, coords, visited, num_cities):
        """
        coords:     (Batch, max_cities, 2)
        visited:    (Batch, max_cities) -> Ignorado en el encoder
        num_cities: (Batch,) -> Cantidad real de ciudades
        """
        B, max_cities, _ = coords.shape
        device = coords.device

        # 1. Creamos la máscara para ignorar el padding en el Transformer
        pad_mask = self._get_pad_mask(max_cities, num_cities, device)

        # 2. Proyectamos y pasamos por el encoder
        enc_input = self.encoder_input_layer(coords)
        
        # src_key_padding_mask exige True en las posiciones que son padding
        memory = self.encoder(enc_input, src_key_padding_mask=pad_mask) 

        return memory

    # =====================================================
    # 2. --- DECODER ---
    # =====================================================
    def decode(self, memory, coords, visited, num_cities):
        """
        memory:     (Batch, max_cities, embed_dim) -> Salida del encoder
        coords:     (Batch, max_cities, 2) -> Ignorado en el decoder
        visited:    (Batch, max_cities) -> Índices ciudades visitadas (-1 para padding)
        num_cities: (Batch,) -> Cantidad real de ciudades
        """
        B, max_cities, _ = memory.shape
        device = memory.device

        # 1. --- MÁSCARA PADDING ---
        pad_mask = self._get_pad_mask(max_cities, num_cities, device)

        # 2. --- MÁSCARA CIUDADES VISITADAS ---
        visited_mask_pos = visited != -1          # (B, max_cities)

        visited_city_mask = torch.zeros(
            B, max_cities, dtype=torch.bool, device=device
        )
        batch_ids, pos_ids = visited_mask_pos.nonzero(as_tuple=True)
        visited_city_mask[batch_ids, visited[batch_ids, pos_ids]] = True

        # Máscara combinada: Prohibido atender a ciudades ya visitadas O que sean padding
        combined_mask = visited_city_mask | pad_mask

        # 3. --- DECODER: Media de ciudades visitadas ---
        # Solo usamos el contexto de las ciudades visitadas reales
        mask_ctx = visited_city_mask.unsqueeze(-1)    # (B, N, 1)

        sum_ctx = (memory * mask_ctx).sum(dim=1)
        count_ctx = mask_ctx.sum(dim=1).clamp(min=1)
        context_mean = sum_ctx / count_ctx            # (B, D)

        # 4. --- DECODER: Primera y última ciudad ---
        start_idx = visited_mask_pos.float().argmax(dim=1)
        last_idx = visited_mask_pos.sum(dim=1) - 1

        batch_idx = torch.arange(B, device=device)
        start_city_embed = memory[
            batch_idx, visited[batch_idx, start_idx].long()
        ]  # (B, D)
        
        last_city_embed = memory[
            batch_idx, visited[batch_idx, last_idx].long()
        ]   # (B, D)

        # Fusión
        ctx_concat = torch.cat(
            [context_mean, last_city_embed, start_city_embed], dim=-1
        )  # (B, 3D)
        decoder_state = self.ctx_fusion(ctx_concat)  # (B, D)

        # 5. --- DECODER: Cross-Attention (Glimpse) ---
        query = self.glimpse_proj(decoder_state).unsqueeze(1)

        for _ in range(self.num_glimpses):
            attn_out, _ = self.cross_attn(
                query=query,            
                key=memory,             
                value=memory,           
                key_padding_mask=combined_mask  # Ignora padding y visitadas
            )

            query = self.norm1(attn_out + query)   
            ff_out = self.ff(query)                
            query = self.norm2(ff_out + query)  

        attn_out = query.squeeze(1)         # (B, D)

        # 6. --- DECODER: Pointer scoring ---
        ptr_query = self.pointer_proj(attn_out)        # (B, D)

        scores = torch.matmul(
            ptr_query.unsqueeze(1),                    
            memory.transpose(1, 2)                     
        ).squeeze(1)                                   # (B, max_cities)

        scores = scores / math.sqrt(self.embed_dim)    

        # 7. --- DECODER: Masking y Softmax ---
        # Aplicamos el infinito negativo tanto a las visitadas como al padding
        scores = scores.masked_fill(combined_mask, -1e9)

        return scores

Para instanciar la red y prepararla para el entrenamiento, debemos definir la configuración de sus hiperparámetros. Estas variables determinan la capacidad, el comportamiento y el costo computacional del modelo:

* `embed_dim`: El tamaño del espacio latente o la "memoria" de la red. Determina qué tan detallados serán los vectores (de 64 dimensiones en este caso) que representan el contexto de cada ciudad.

* `num_heads`: La cantidad de cabezas en el mecanismo de atención. Al usar 4 cabezas, permitimos que el modelo analice simultáneamente distintas perspectivas del grafo (por ejemplo, prestando atención al mismo tiempo a vecinos muy cercanos y a la densidad general de una zona más alejada).

* `num_encoder_layers`: La profundidad del *encoder*. Define cuántos bloques de atención consecutiva procesarán el mapa. Con 2 capas, las representaciones de las ciudades se refinarán dos veces interactuando entre sí antes de enviarse al decodificador.

* `num_glimpses`: El núcleo de esta arquitectura. Establece la cantidad de iteraciones de *Cross-Attention* que realizará el decodificador. Con 2 vistazos, la red consulta el mapa completo dos veces seguidas para recalibrar su estado interno antes de decidir el siguiente paso.

* `dropout`: La tasa de regularización. Apaga aleatoriamente el 10% de las conexiones neuronales en cada paso del entrenamiento. Esto evita que la red memorice rutas específicas y la fuerza a aprender patrones generales de resolución.

In [7]:
from models.glimpse import TSPTransformer

input_dim = 2 # Dimensión de las características de entrada, en este caso 2 (X, Y)

# Parámetros del modelo
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
model = TSPTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout
)

## Entrenamiento y validación

Para entrenar nuestro modelo Glimpse, replicaremos el mismo procedimiento detallado en el notebook introductorio. Primero, dividiremos nuestro conjunto de datos generado en una proporción de 80/20 para obtener los subconjuntos de entrenamiento y validación. Posteriormente, ejecutaremos el ciclo de aprendizaje supervisado optimizando la función de pérdida *Cross Entropy* y monitoreando la métrica de *Accuracy* a lo largo de 10 épocas.

In [8]:
from data.preprocessing import load_dataset, split_dataset

# Split 80/20
train_file, val_file = split_dataset("train_data.h5", 4000)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

model = sl_train(
    model=model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    loss_fn=CrossEntropyLoss(),
    metrics=[Accuracy()],
    metrics_filename="metrics.txt"
)

Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (4000 muestras) en: train_data_train.h5
Guardando Val puro (1000 muestras) en: train_data_val.h5
Dataset train_data_train.h5 cargado con 4000 muestras.
Dataset train_data_val.h5 cargado con 1000 muestras.
** Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 2.3765 | Val CrossEntropy: 1.4800
    Accuracy: 60.90%
Epoch 2/10
    Train CrossEntropy: 1.4367 | Val CrossEntropy: 1.1510
    Accuracy: 71.30%
Epoch 3/10
    Train CrossEntropy: 1.0937 | Val CrossEntropy: 0.8944
    Accuracy: 79.00%
Epoch 4/10
    Train CrossEntropy: 0.9569 | Val CrossEntropy: 0.8320
    Accuracy: 79.20%
Epoch 5/10
    Train CrossEntropy: 0.8933 | Val CrossEntropy: 0.7627
    Accuracy: 80.70%
Epoch 6/10
    Train CrossEntropy: 0.8295 | Val CrossEntropy: 0.7192
    Accuracy: 80.70%
Epoch 7/10
    Train CrossEntropy: 0.7955 | Val CrossEntropy: 0.7080
    Accuracy: 80.90%
Epoch 8/10
    Train CrossEntropy

Para validar el modelo, utilizaremos la función `evaluate`, que se encarga de calcular directamente los costos promedios obtenidos por el modelo y OR-Tools, además de calcular el gap.

In [14]:
from solvers.eval import evaluate
from data.adapters.input.default import DefaultInputAdapter

# 1. Definimos la misma configuración del adaptador que usamos en entrenamiento
input_config = (DefaultInputAdapter, 50)

# 2. Ejecutamos la evaluación paralela sobre un conjunto de prueba no visto
# La función evaluate calculará e imprimirá automáticamente los costos, el gap y la desviación estándar.
model_sols, ort_sols = evaluate(
    model=model,
    instance_file="benchmarks/B50.pkl",
    input_adapter_config=input_config,
    num_workers=None  # Utiliza todos los núcleos de CPU disponibles
)

Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.65
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      14.36% ± 6.96%

